# Explainable Career Decision Support System

**Final project — CS 4580/5580 Automated Decision Systems**  
Kevin Rusagara Iraguha

This notebook walks through an automated decision system that recommends
career paths from a structured user profile. Unlike a black-box predictor,
every recommendation comes with a **decision trace**: which rules fired,
which profile attributes triggered them, and how each one contributed to
the score. Four explanation modes make the reasoning fully inspectable:

1. **Top reasons** — positive and negative factors per career
2. **Head-to-head** — what differentiates #1 from #2
3. **Counterfactuals** — minimal profile changes that flip the recommendation
4. **Tradeoffs** — rules that pull in opposing directions

Plus an **option-generation** layer that surfaces hybrid roles, dark-horse
options, and weak-match warnings.

## 1. System overview

In [1]:
from advisor import (
    CAREERS, RULES, UserProfile, score,
    explain_top, head_to_head, counterfactuals, detect_tradeoffs, alternatives,
)
from advisor.scoring import ranked
from sample_profiles import (
    swe_candidate, research_candidate, consulting_candidate,
    conflicted_candidate, early_career_candidate,
)

print(f'Career options in catalog: {len(CAREERS)}')
print(f'Rules in rule base:        {len(RULES)}')
print()
print('Careers:')
for c in CAREERS:
    print(f'  - {c.name}: {c.description}')

Career options in catalog: 13
Rules in rule base:        40

Careers:
  - Software Engineering: Designing and building software products and systems.
  - Data Science: Extracting insights and predictions from data using statistics and ML.
  - Machine Learning Engineering: Building production ML systems, pipelines, and infrastructure.
  - Product Management: Owning product strategy, prioritization, and cross-functional execution.
  - UX / Product Design: Researching users and designing interfaces and experiences.
  - Management Consulting: Advising organizations on strategy, operations, and transformation.
  - Quantitative Finance: Quantitative research, trading, or risk in financial markets.
  - Industry Research: Applied research at scale in industry R&D labs.
  - Academic Research: PhD-track research, publications, and faculty paths.
  - DevOps / SRE: Reliability, infrastructure, and developer-platform engineering.
  - Cybersecurity: Securing systems — defensive, offensive, or govern

## 2. Profile A — clear software-engineering candidate

Strong programming skills, prior SWE internship, building / shipping interests,
and a long-term goal of starting a company. We'd expect SWE and entrepreneurship
to dominate, with the goal of starting a company tilting things toward founder paths.

In [2]:
profile = swe_candidate()
print(profile.summary())

Profile: Alex (SWE-leaning senior)
  Technical: python(5), java(4), javascript(4), system_design(4), linux(4)
  Non-technical: communication(3), writing(3), leadership(2)
  Interests: building, products, ai
  Work style: hands_on, collaborative, fast_paced
  Short-term goals: land a high-impact engineering role at a product company; earn a competitive starting salary
  Long-term goals: become a tech lead and eventually start a company


In [3]:
scores, firings = score(profile)
print(explain_top(scores, firings, top_n=3))

TOP RECOMMENDATIONS

#1  Entrepreneurship   (score: +10.50)
    Why this ranks high:
      +4.00  Wanting to start a company favors the founder path.
             - goal mentions 'start a company'
      +2.50  Fast-paced preference suits startups and consulting sprints.
             - work style: fast_paced
      +2.00  Loving to build products favors engineering and founder paths.
             - interest: building
             - interest: products
      +1.50  Wanting impact / mission alignment supports research and founder paths.
             - goal mentions 'impact'
      +0.50  Strong general-purpose programming supports engineering roles.
             - python: 5/5
             - java: 4/5
             - javascript: 4/5

#2  Software Engineering   (score: +9.97)
    Why this ranks high:
      +3.00  Strong general-purpose programming supports engineering roles.
             - python: 5/5
             - java: 4/5
             - javascript: 4/5
      +2.00  Loving to build products 

In [4]:
print(alternatives(profile, scores, firings))

ALTERNATIVE OPTIONS
  CLOSE CALL: Entrepreneurship (+10.50) and Software Engineering (+9.97) are very close.
     Consider: Technical founder
  DARK HORSE: Product Management has substantial positive support (+6.00 of positive evidence) but is held back by penalties. Worth considering if those constraints can be addressed.


In [5]:
top_two = ranked(scores)[:2]
print(head_to_head(top_two[0][0], top_two[1][0], firings))

HEAD-TO-HEAD: Entrepreneurship  vs  Software Engineering
  Δ +3.50  (+4.00 vs +0.50)  →  favors Entrepreneurship
             Wanting to start a company favors the founder path.
  Δ -2.50  (+0.50 vs +3.00)  →  favors Software Engineering
             Strong general-purpose programming supports engineering roles.
  Δ +2.50  (+2.50 vs +0.00)  →  favors Entrepreneurship
             Fast-paced preference suits startups and consulting sprints.
  Δ -2.00  (+0.00 vs +2.00)  →  favors Software Engineering
             Hands-on builders prefer engineering-track roles.
  Δ -1.67  (+0.00 vs +1.67)  →  favors Software Engineering
             Prior software engineering experience strengthens SWE recruiting.
  Δ +1.50  (+1.50 vs +0.00)  →  favors Entrepreneurship
             Wanting impact / mission alignment supports research and founder paths.
  Δ -0.80  (+0.00 vs +0.80)  →  favors Software Engineering
             Systems / infrastructure skills support reliability and security work.


In [6]:
print(counterfactuals(profile, scores))

COUNTERFACTUALS — what would change the recommendation?
  If you raised statistics to 4/5, top recommendation would shift to Machine Learning Engineering.


**What to notice.** The decision trace shows *exactly* which rules fired and which
profile attributes triggered them. The head-to-head explains the small gap between
the top two careers as a tradeoff between founder-direction goals and concrete SWE
evidence. The counterfactual is the system's way of saying: *here is a small
profile change that would change my mind.*

## 3. Profile B — research-track candidate

Deep ML / math background, research experience, an explicit PhD goal. The system
should land decisively on academic research, with industry research as a strong
second.

In [7]:
profile = research_candidate()
print(profile.summary())
scores, firings = score(profile)
print()
print(explain_top(scores, firings, top_n=3))

Profile: Priya (research-track)
  Technical: machine_learning(5), deep_learning(5), math(5), statistics(5), python(4)
  Non-technical: writing(5), communication(3)
  Interests: research, ai, open_problems, theory
  Work style: independent, strategic
  Short-term goals: apply to top PhD programs in machine learning
  Long-term goals: pursue a PhD and contribute to fundamental AI research

TOP RECOMMENDATIONS

#1  Academic Research   (score: +17.50)
    Why this ranks high:
      +4.00  Wanting a PhD strongly favors the academic track.
             - goal mentions 'phd'
      +3.00  Curiosity about open problems fits research-oriented careers.
             - interest: research
             - interest: open_problems
             - interest: theory
      +2.00  Strong statistics / math underpins data and quantitative work.
             - statistics: 5/5
             - math: 5/5
             - linear algebra: 4/5
      +2.00  Strong writing is leveraged in research, PM specs, and consulting

In [8]:
print(counterfactuals(profile, scores))

COUNTERFACTUALS — what would change the recommendation?
  The top recommendation is robust — no single small change to the profile would flip it.


**What to notice.** When the top recommendation is well-supported by many
independent rules, the counterfactual search reports that no small profile
change would flip it — i.e., the recommendation is *robust*. That's an
important property of an explainable system.

## 4. Profile C — consulting-aspiring candidate (with a constraint twist)

Strong communication and strategic thinking, an explicit consulting goal —
**but** a `no_relocation` constraint. Consulting jobs are concentrated in major
cities, so the constraint penalizes consulting. We'd expect the system to surface
this as a tradeoff.

In [9]:
profile = consulting_candidate()
print(profile.summary())
scores, firings = score(profile)
print()
print(explain_top(scores, firings, top_n=3))

Profile: Jordan (strategy / consulting)
  Technical: math(3), statistics(3), python(2)
  Non-technical: communication(5), presentation(5), writing(4), leadership(4)
  Interests: strategy, business, users
  Work style: collaborative, fast_paced, strategic, ambiguous
  Short-term goals: land an offer at a top management consulting firm
  Long-term goals: move into product strategy or general management
  Constraints: no_relocation

TOP RECOMMENDATIONS

#1  Product Management   (score: +16.20)
    Why this ranks high:
      +3.00  Strong communication is critical for client-facing and cross-functional roles.
             - communication: 5/5
             - presentation: 5/5
      +2.50  Strategic thinkers gravitate to product and consulting roles.
             - work style: strategic
      +2.00  Interest in strategy fits consulting and product roles.
             - interest: strategy
             - interest: business
      +2.00  Interest in users / human factors supports UX and PM.
    

In [10]:
top_two = ranked(scores)[:2]
print(head_to_head(top_two[0][0], top_two[1][0], firings))

HEAD-TO-HEAD: Product Management  vs  Management Consulting
  Δ +2.00  (+2.00 vs +0.00)  →  favors Product Management
             Interest in users / human factors supports UX and PM.
  Δ +2.00  (+0.00 vs -2.00)  →  favors Product Management
             Inability to relocate cuts options concentrated in specific cities.
  Δ -1.00  (+2.00 vs +3.00)  →  favors Management Consulting
             Interest in strategy fits consulting and product roles.
  Δ -1.00  (+1.00 vs +2.00)  →  favors Management Consulting
             Comfort with ambiguity is a strong fit for founder and consultant roles.
  Δ -0.50  (+1.50 vs +2.00)  →  favors Management Consulting
             Collaborative preference suits cross-functional roles.
  Δ -0.50  (+1.00 vs +1.50)  →  favors Management Consulting
             Fast-paced preference suits startups and consulting sprints.
  Δ +0.50  (+2.50 vs +2.00)  →  favors Product Management
             Strategic thinkers gravitate to product and consulting roles.
  

In [11]:
print(counterfactuals(profile, scores))

COUNTERFACTUALS — what would change the recommendation?
  If the constraint 'no_relocation' were removed, top recommendation would shift to Management Consulting.


**What to notice.** The user *said* they wanted consulting, but the system
reports that PM ranks higher *because* of the relocation constraint. That's
exactly the kind of insight a black-box recommender would hide. The
counterfactual then says: *if you removed the no_relocation constraint, the
recommendation would shift back to consulting.* The user gets the full picture
and can decide for themselves whether to relax the constraint.

## 5. Profile D — conflicted candidate (skills vs. preferences)

Strong programming background plus solid communication and leadership. The user
wants a role combining technical depth with product impact. This is the kind of
profile where the alternative-options engine adds the most value.

In [12]:
profile = conflicted_candidate()
print(profile.summary())
scores, firings = score(profile)
print()
print(explain_top(scores, firings, top_n=3))

Profile: Sam (conflicted: SWE skills, PM aspirations)
  Technical: python(5), java(4), system_design(3), machine_learning(3)
  Non-technical: communication(4), presentation(4), leadership(4), writing(3)
  Interests: building, products, strategy, users
  Work style: collaborative, strategic
  Short-term goals: find a role that combines technical depth with product impact
  Long-term goals: become a product leader at a major tech company

TOP RECOMMENDATIONS

#1  Product Management   (score: +14.50)
    Why this ranks high:
      +2.50  Strategic thinkers gravitate to product and consulting roles.
             - work style: strategic
      +2.40  Strong communication is critical for client-facing and cross-functional roles.
             - communication: 4/5
             - presentation: 4/5
      +2.00  Interest in strategy fits consulting and product roles.
             - interest: strategy
      +2.00  Interest in users / human factors supports UX and PM.
             - interest: users


In [13]:
print(alternatives(profile, scores, firings))

ALTERNATIVE OPTIONS
  DARK HORSE: Technical Program Management has substantial positive support (+7.90 of positive evidence) but is held back by penalties. Worth considering if those constraints can be addressed.


## 6. Profile E — early-career, weak signal

Few skills, no experience, no strong goals. We expect every score to be low —
the system should flag that the profile doesn't yet provide enough signal to
make a confident recommendation.

In [14]:
profile = early_career_candidate()
print(profile.summary())
scores, firings = score(profile)
print()
print(explain_top(scores, firings, top_n=3))

Profile: Morgan (exploring options)
  Technical: python(2)
  Non-technical: communication(3)
  Interests: ai
  Work style: collaborative
  Short-term goals: explore what kinds of work I might enjoy
  Constraints: no_phd

TOP RECOMMENDATIONS

#1  Management Consulting   (score: +2.00)
    Why this ranks high:
      +2.00  Collaborative preference suits cross-functional roles.
             - work style: collaborative

#2  Technical Program Management   (score: +2.00)
    Why this ranks high:
      +2.00  Collaborative preference suits cross-functional roles.
             - work style: collaborative

#3  Product Management   (score: +1.50)
    Why this ranks high:
      +1.50  Collaborative preference suits cross-functional roles.
             - work style: collaborative


In [15]:
print(alternatives(profile, scores, firings))

ALTERNATIVE OPTIONS
  CLOSE CALL: Management Consulting (+2.00) and Technical Program Management (+2.00) are very close.
     Consider: Tech-strategy consulting / transformation lead
  WEAK MATCH: even the top option (Management Consulting) scores only +2.00.
     The profile may need more skill or experience signal before any of these careers becomes a strong fit. Use the counterfactuals below to see which additions would most change the picture.


In [16]:
print(counterfactuals(profile, scores))

COUNTERFACTUALS — what would change the recommendation?
  If you raised python to 4/5, top recommendation would shift to Software Engineering.
  If you raised machine learning to 4/5, top recommendation would shift to Machine Learning Engineering.
  If you raised statistics to 4/5, top recommendation would shift to Data Science.
  If you raised system design to 4/5, top recommendation would shift to DevOps / SRE.
  If you raised security to 4/5, top recommendation would shift to Cybersecurity.


**What to notice.** The system explicitly flags this as a *weak match* rather
than recommending the highest-scoring option as if it were a confident pick.
The counterfactuals here become *guidance* — concrete skills the user could
develop to make any career a stronger fit.

## 7. Tradeoff inspection

Some rules push one career up while pushing another down. The tradeoff inspector
surfaces these — useful for understanding *why* a particular profile constrains
the option space.

In [17]:
profile = research_candidate()
scores, firings = score(profile)
print(detect_tradeoffs(firings))

TRADEOFFS — rules that pull in opposing directions
  No active rules create opposing pressures on careers.


## 8. Build your own profile

Edit the cell below to construct a custom profile and run the full pipeline.
Skill levels are 0-5. Common interests, work-styles, and constraints are listed
in `advisor/rules.py`.

In [18]:
custom = UserProfile(
    name='Custom user',
    technical_skills={
        'python': 4,
        'machine_learning': 3,
        'statistics': 4,
    },
    non_technical_skills={
        'communication': 4,
        'writing': 3,
    },
    interests=['ai', 'building', 'users'],
    experience=[
        {'field': 'software_engineering', 'role': 'intern', 'years': 1},
    ],
    work_style=['collaborative', 'strategic'],
    short_term_goals=['find a role with both technical depth and product impact'],
    long_term_goals=['become a tech lead'],
    constraints=[],
)

scores, firings = score(custom)
print(custom.summary())
print()
print(explain_top(scores, firings, top_n=3))
print()
print(alternatives(custom, scores, firings))
print()
print(counterfactuals(custom, scores))

Profile: Custom user
  Technical: python(4), statistics(4), machine_learning(3)
  Non-technical: communication(4), writing(3)
  Interests: ai, building, users
  Work style: collaborative, strategic
  Short-term goals: find a role with both technical depth and product impact
  Long-term goals: become a tech lead

TOP RECOMMENDATIONS

#1  Product Management   (score: +10.90)
    Why this ranks high:
      +2.50  Strategic thinkers gravitate to product and consulting roles.
             - work style: strategic
      +2.40  Strong communication is critical for client-facing and cross-functional roles.
             - communication: 4/5
      +2.00  Interest in users / human factors supports UX and PM.
             - interest: users
      +1.50  Loving to build products favors engineering and founder paths.
             - interest: building
      +1.50  Collaborative preference suits cross-functional roles.
             - work style: collaborative

#2  Machine Learning Engineering   (score: 

## 9. How the system explains its decisions

Every score in this system is fully traceable back to:

1. **A specific rule** in `advisor/rules.py` (with a human-readable description).
2. **A condition** that fired against the user's profile.
3. **The exact profile attributes** (skills, interests, work style, goals,
   constraints) that satisfied that condition — surfaced as `evidence` strings.
4. **A signed numeric contribution** to one or more career scores.

This is the opposite of a neural-network classifier: there is no opaque
weight matrix, no embedding, no black box. If the user disagrees with a
recommendation, they can read the trace and identify which specific rule
they disagree with — and the system tells them what would have to change
for the recommendation to flip.

The four explanation modes serve four distinct user questions:

- *"Why did you recommend this?"* → **`explain_top`**
- *"Why this one over that one?"* → **`head_to_head`**
- *"What would change your mind?"* → **`counterfactuals`**
- *"What's pulling against itself in my profile?"* → **`detect_tradeoffs`**

And `alternatives` adds a fifth: *"What other options should I consider?"* —
covering hybrid roles, dark horses, and weak-match warnings.